# Final Task B 17-class - Swin-Tiny

Train Swin-Tiny directly on the final 17 species classes. This uses ImageNet pretrained initialization and does not load the old 20-class Task B checkpoints.


In [2]:
from pathlib import Path
import sys

if Path('/content').exists():
    ROOT = Path('/content/woodbench')
else:
    ROOT = next((parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / 'src').exists()), Path.cwd())

sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
FINAL_B_DIR = ROOT / "notebooks" / "phase 1" / "task-B-17-new"
sys.path.insert(0, str(FINAL_B_DIR))
print(f"ROOT: {ROOT}")
print(f"task-B-17-new: {FINAL_B_DIR}")


ROOT: /home/sakana/Code/ManifoldsTeam/woodbench
task-B-17-new: /home/sakana/Code/ManifoldsTeam/woodbench/notebooks/phase 1/task-B-17-new


In [2]:
# Colab-only setup. Cloud data is expected to be already converted to 17 classes
# by 00_data_preparation.ipynb. This download only runs when the dataset is absent.
if Path('/content').exists():
    %pip install -q -r {ROOT}/requirements.txt

    DATA_DIR = ROOT / "data" / "task-B-group-split"
    if not DATA_DIR.exists():
        from huggingface_hub import login, snapshot_download
        import os

        token = os.getenv("HF_TOKEN", "")
        if token:
            login(token=token)
        snapshot_download(
            repo_id="ny0g507h3p/wood-task-B",
            repo_type="dataset",
            local_dir=str(DATA_DIR),
            ignore_patterns=["*.md", "README*"],
        )


In [3]:
from final_b_17class import (
    BACKBONE_CONFIGS,
    prepare_environment,
    run_all_experiments,
    summarize_results,
    verify_17class_dataset,
)

prepare_environment(ROOT)

BACKBONE = "swin_tiny"
DISPLAY_NAME = "Swin-Tiny"
DATA_ROOT = ROOT / "data" / "task-B-group-split"
BATCH_SIZE = BACKBONE_CONFIGS[BACKBONE]["batch_size"]
EPOCHS = 50

print(f"Backbone: {DISPLAY_NAME} ({BACKBONE})")
print(f"Batch size: {BATCH_SIZE}")
print(f"Data root: {DATA_ROOT}")


Backbone: Swin-Tiny (swin_tiny)
Batch size: 32
Data root: /home/sakana/Code/ManifoldsTeam/woodbench/data/task-B-group-split


In [3]:
class_names = verify_17class_dataset(DATA_ROOT)
print(f"Classes ({len(class_names)}):")
for idx, class_name in enumerate(class_names):
    print(f"  {idx:02d}: {class_name}")


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Classes (17):
  00: Araucaria araucana
  01: Callitris columellaris
  02: Calocedrus rupestris
  03: Chamaecyparis obtusa
  04: Cryptomeria japonica
  05: Cunninghamia lanceolata
  06: Fokienia hodginsii
  07: Picea glauca
  08: Pinus cernua
  09: Pinus dalatensis
  10: Pinus krempfii
  11: Pinus radiata
  12: Podocarpus pilgeri
  13: Pseudotsuga brevifolia
  14: Taxus wallichiana
  15: Thujopsis dolabrata
  16: Xanthocyparis vietnamensis


In [ ]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")

results = run_all_experiments(
    backbone_name=BACKBONE,
    root=ROOT,
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    device=DEVICE,
)


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB
[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
[Device] cuda
[Output] /workspace/woodbench/data/experiments/Task-B-17class

main/seed_42/swin_tiny
[main | swin_tiny s42] train=3365, val=903, test=532, classes=17


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Epoch 1/50 | Train Loss: 2.8548, Acc: 6.52% | Val Loss: 2.8385, Acc: 5.87% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 5.87%)


Epoch 2/50 | Train Loss: 1.7006, Acc: 63.21% | Val Loss: 0.8254, Acc: 94.13% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 94.13%)


Epoch 3/50 | Train Loss: 0.7720, Acc: 95.80% | Val Loss: 0.6800, Acc: 97.45% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 97.45%)


Epoch 4/50 | Train Loss: 0.6653, Acc: 98.45% | Val Loss: 0.6509, Acc: 98.23% | LR: 8.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 98.23%)


... (98 outputs left out)

Epoch 1/50 | Train Loss: 2.8584, Acc: 6.52% | Val Loss: 2.8450, Acc: 5.87% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 5.87%)


Epoch 2/50 | Train Loss: 1.9276, Acc: 63.12% | Val Loss: 1.2559, Acc: 94.13% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 94.13%)


Epoch 3/50 | Train Loss: 1.1882, Acc: 96.07% | Val Loss: 1.1298, Acc: 95.90% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 95.90%)


Epoch 4/50 [Train]:  65%|██████▍   | 68/105 [02:18<00:45,  1.23s/batch, loss=1.0605, acc=98.3%]

In [ ]:
summarize_results(results)


## Continue after Vast interruption

Run the root, import/config, and data-check cells above, but do not rerun the old `run_all_experiments` cell. Then run each config cell below independently. Completed metrics are reused; interrupted checkpoints restore model, optimizer, and scheduler state.

In [6]:
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("CUDA not available; CPU training will be slow.")


GPU: NVIDIA GeForce RTX 4090
VRAM: 23.5 GB


In [7]:
from final_b_17class import EXPERIMENTS, load_completed_results, train_one

continued_results = {
    (row["experiment"], row["seed"]): row
    for row in load_completed_results(backbone_name=BACKBONE, root=ROOT)
}

def run_remaining(experiment_name, seed):
    result = train_one(
        root=ROOT,
        backbone=BACKBONE,
        seed=seed,
        experiment_name=experiment_name,
        experiment_cfg=EXPERIMENTS[experiment_name],
        data_root=DATA_ROOT,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        device=DEVICE,
        resume_training=True,
    )
    continued_results[(experiment_name, seed)] = result
    return result

print(f"Loaded {len(continued_results)}/6 completed configs for {BACKBONE}.")


Loaded 6/6 completed configs for swin_tiny.


In [7]:
# Resume label-smoothing=0.2 from its saved checkpoint
run_remaining("taskB_label_smoothing_0.2", 42)


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


[taskB_label_smoothing_0.2 | swin_tiny s42] train=3365, val=903, test=532, classes=17
[Resume] /workspace/woodbench/data/experiments/Task-B-17class/taskB_label_smoothing_0.2/swin_tiny/swin_tiny_seed42_best.pth -> epoch 8/50


Epoch 8/50 | Train Loss: 1.0381, Acc: 99.32% | Val Loss: 1.0267, Acc: 99.56% | LR: 9.89e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 99.56%)


Epoch 9/50 | Train Loss: 1.0196, Acc: 99.91% | Val Loss: 1.0370, Acc: 98.89% | LR: 9.81e-05[EarlyStop 1/10]


Epoch 10/50 | Train Loss: 1.0227, Acc: 99.67% | Val Loss: 1.0222, Acc: 99.34% | LR: 9.70e-05[EarlyStop 2/10]


Epoch 11/50 | Train Loss: 1.0200, Acc: 99.73% | Val Loss: 1.0157, Acc: 99.78% | LR: 9.57e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed42_best.pth (Val Acc: 99.78%)


... (20 outputs left out)

Epoch 33/50 | Train Loss: 1.0118, Acc: 99.82% | Val Loss: 1.0223, Acc: 99.22% | LR: 3.13e-05[EarlyStop 8/10]


Epoch 34/50 | Train Loss: 1.0079, Acc: 100.00% | Val Loss: 1.0116, Acc: 99.78% | LR: 2.81e-05[EarlyStop 9/10]


Epoch 35/50 | Train Loss: 1.0075, Acc: 100.00% | Val Loss: 1.0129, Acc: 99.78% | LR: 2.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 35.
[taskB_label_smoothing_0.2 | swin_tiny s42] val_acc=100.00% test_acc=99.44% F1=99.49%


{'backbone': 'swin_tiny',
 'seed': 42,
 'experiment': 'taskB_label_smoothing_0.2',
 'num_classes': 17,
 'class_names': ['Araucaria araucana',
  'Callitris columellaris',
  'Calocedrus rupestris',
  'Chamaecyparis obtusa',
  'Cryptomeria japonica',
  'Cunninghamia lanceolata',
  'Fokienia hodginsii',
  'Picea glauca',
  'Pinus cernua',
  'Pinus dalatensis',
  'Pinus krempfii',
  'Pinus radiata',
  'Podocarpus pilgeri',
  'Pseudotsuga brevifolia',
  'Taxus wallichiana',
  'Thujopsis dolabrata',
  'Xanthocyparis vietnamensis'],
 'best_val_acc': 100.0,
 'best_epoch': 25,
 'final_train_acc': 100.0,
 'final_val_acc': 99.77851605758582,
 'stopped_early': True,
 'epochs_trained': 35,
 'test_accuracy': 0.9943609022556391,
 'test_f1_macro': 0.9948557461350482,
 'test_precision_macro': 0.9948561242678889,
 'test_recall_macro': 0.9950980392156863,
 'test_f1_per_class': {'Araucaria araucana': 1.0,
  'Callitris columellaris': 1.0,
  'Calocedrus rupestris': 1.0,
  'Chamaecyparis obtusa': 0.9705882352

In [8]:
results = list(continued_results.values())
summarize_results(results)



Task B 17-class summary
experiment                       seed   val_acc  test_acc        f1      prec       rec
----------------------------------------------------------------------------------------
main                               42    99.67%    97.18%    97.39%    97.99%    97.31%
main                              123    99.78%   100.00%   100.00%   100.00%   100.00%
main                              456    99.67%    99.25%    99.28%    99.34%    99.28%
taskB_90rotation                   42   100.00%   100.00%   100.00%   100.00%   100.00%
taskB_label_smoothing_0.0          42    99.56%    99.62%    99.65%    99.65%    99.67%
taskB_label_smoothing_0.2          42   100.00%    99.44%    99.49%    99.49%    99.51%
----------------------------------------------------------------------------------------
main test_acc: 98.81 +/- 1.19%
main test_f1:  98.89 +/- 1.10%


In [4]:
# Fresh audit runs: main seed 123, 90-degree rotation, and label smoothing 0.2.
# Run this cell after the ROOT/import/data-check cells. Outputs are isolated from official checkpoints.
import gc
import json
from pathlib import Path

import pandas as pd
import torch

import final_b_17class as final_b

AUDIT_BACKBONE = "swin_tiny"
AUDIT_BATCH_SIZE = final_b.BACKBONE_CONFIGS[AUDIT_BACKBONE]["batch_size"]
AUDIT_DATA_ROOT = ROOT / "data" / "task-B-group-split"
AUDIT_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
AUDIT_EPOCHS = 50
AUDIT_RUN_NAME = "swin_val100_recheck_v1"
AUDIT_OUTPUT_ROOT = (
    Path("data") / "experiments" / "Task-B-new" / AUDIT_RUN_NAME
)
REFERENCE_METRICS_PATH = (
    ROOT / "data" / "experiments" / "phase_1" / "final_b_recomputed" / "split_metrics.csv"
)

AUDIT_CONFIGS = [
    {
        "config": "main_seed123",
        "source_experiment": "main",
        "experiment_name": "recheck_main_seed123",
        "seed": 123,
        "label_smoothing": 0.1,
        "use_90rotation": False,
    },
    {
        "config": "rotation_90_seed42",
        "source_experiment": "taskB_90rotation",
        "experiment_name": "recheck_90rotation_seed42",
        "seed": 42,
        "label_smoothing": 0.1,
        "use_90rotation": True,
    },
    {
        "config": "label_smoothing_0.2_seed42",
        "source_experiment": "taskB_label_smoothing_0.2",
        "experiment_name": "recheck_label_smoothing_0.2_seed42",
        "seed": 42,
        "label_smoothing": 0.2,
        "use_90rotation": False,
    },
]

final_b.prepare_environment(ROOT)
audit_classes = final_b.verify_17class_dataset(AUDIT_DATA_ROOT)
assert len(audit_classes) == 17
print(f"Audit device: {AUDIT_DEVICE}")
print(f"Isolated output: {ROOT / AUDIT_OUTPUT_ROOT}")
print("Existing runs in this audit namespace resume/reuse; official checkpoints are never touched.")

reference_metrics = None
if REFERENCE_METRICS_PATH.exists():
    reference_metrics = pd.read_csv(REFERENCE_METRICS_PATH)
    print(f"Reference metrics: {REFERENCE_METRICS_PATH}")
else:
    print("Reference split_metrics.csv not found; deltas will be NaN.")

audit_rows = []
audit_details = []
original_output_root = final_b.OUTPUT_EXPERIMENT_ROOT
try:
    final_b.OUTPUT_EXPERIMENT_ROOT = AUDIT_OUTPUT_ROOT

    for spec in AUDIT_CONFIGS:
        experiment_cfg = {
            "label_smoothing": spec["label_smoothing"],
            "use_90rotation": spec["use_90rotation"],
            "tags": ["audit", "val100-recheck", spec["config"]],
        }
        print("\n" + "=" * 96)
        print(f"Fresh audit config: {spec['config']}")
        print("=" * 96)

        training_result = final_b.train_one(
            root=ROOT,
            backbone=AUDIT_BACKBONE,
            seed=spec["seed"],
            experiment_name=spec["experiment_name"],
            experiment_cfg=experiment_cfg,
            data_root=AUDIT_DATA_ROOT,
            batch_size=AUDIT_BATCH_SIZE,
            epochs=AUDIT_EPOCHS,
            device=AUDIT_DEVICE,
            resume_training=True,
        )

        output_dir = final_b.experiment_output_dir(
            ROOT, spec["experiment_name"], spec["seed"], AUDIT_BACKBONE
        )
        checkpoint_path = output_dir / f"{AUDIT_BACKBONE}_seed{spec['seed']}_best.pth"
        assert checkpoint_path.exists(), f"Missing best checkpoint: {checkpoint_path}"

        train_loader, val_loader, test_loader = final_b.get_dataloaders(
            data_root=AUDIT_DATA_ROOT,
            batch_size=AUDIT_BATCH_SIZE,
            num_workers=4,
            task="taskB",
            use_90rotation=spec["use_90rotation"],
        )
        checkpoint = final_b._load_checkpoint(checkpoint_path)
        best_model = final_b.create_model(
            AUDIT_BACKBONE, num_classes=len(audit_classes), pretrained=False
        )
        best_model.load_state_dict(checkpoint["model_state_dict"], strict=True)
        best_model = best_model.to(AUDIT_DEVICE).eval()

        val_metrics = final_b.evaluate_model(best_model, val_loader, AUDIT_DEVICE)
        test_metrics = final_b.evaluate_model(best_model, test_loader, AUDIT_DEVICE)

        row = {
            "config": spec["config"],
            "seed": spec["seed"],
            "label_smoothing": spec["label_smoothing"],
            "use_90rotation": spec["use_90rotation"],
            "reported_best_val_accuracy_pct": training_result["best_val_acc"],
            "best_checkpoint_val_accuracy": val_metrics["test_accuracy"],
            "best_checkpoint_val_f1_macro": val_metrics["test_f1_macro"],
            "best_checkpoint_test_accuracy": test_metrics["test_accuracy"],
            "best_checkpoint_test_f1_macro": test_metrics["test_f1_macro"],
            "val_test_accuracy_gap_pp": (
                val_metrics["test_accuracy"] - test_metrics["test_accuracy"]
            ) * 100,
            "best_epoch": training_result["best_epoch"],
            "epochs_trained": training_result["epochs_trained"],
            "checkpoint": str(checkpoint_path.relative_to(ROOT)),
        }

        if reference_metrics is not None:
            for split in ["val", "test"]:
                matched = reference_metrics[
                    (reference_metrics["experiment"] == spec["source_experiment"])
                    & (reference_metrics["seed"] == spec["seed"])
                    & (reference_metrics["backbone"] == AUDIT_BACKBONE)
                    & (reference_metrics["split"] == split)
                ]
                assert len(matched) == 1, (spec["config"], split, len(matched))
                row[f"reference_{split}_accuracy"] = float(matched.iloc[0]["accuracy"])
                row[f"reference_{split}_f1_macro"] = float(matched.iloc[0]["f1_macro"])
            row["test_accuracy_delta_vs_reference_pp"] = (
                row["best_checkpoint_test_accuracy"] - row["reference_test_accuracy"]
            ) * 100

        audit_rows.append(row)
        audit_details.append(
            {
                **row,
                "training_result": training_result,
                "best_checkpoint_val_metrics": val_metrics,
                "best_checkpoint_test_metrics": test_metrics,
            }
        )

        del best_model, checkpoint, train_loader, val_loader, test_loader
        gc.collect()
        if AUDIT_DEVICE == "cuda":
            torch.cuda.empty_cache()
finally:
    final_b.OUTPUT_EXPERIMENT_ROOT = original_output_root

audit_output_dir = ROOT / AUDIT_OUTPUT_ROOT
audit_output_dir.mkdir(parents=True, exist_ok=True)
audit_comparison = pd.DataFrame(audit_rows)
audit_comparison.to_csv(audit_output_dir / "best_checkpoint_comparison.csv", index=False)
(audit_output_dir / "best_checkpoint_metrics.json").write_text(
    json.dumps(audit_details, indent=2, ensure_ascii=False), encoding="utf-8"
)

display_columns = [
    "config",
    "seed",
    "reported_best_val_accuracy_pct",
    "best_checkpoint_val_accuracy",
    "best_checkpoint_test_accuracy",
    "best_checkpoint_test_f1_macro",
    "val_test_accuracy_gap_pp",
]
if "test_accuracy_delta_vs_reference_pp" in audit_comparison:
    display_columns.append("test_accuracy_delta_vs_reference_pp")

audit_display = audit_comparison[display_columns].copy()
for column in [
    "best_checkpoint_val_accuracy",
    "best_checkpoint_test_accuracy",
    "best_checkpoint_test_f1_macro",
]:
    audit_display[column] *= 100
print(f"Saved audit comparison: {audit_output_dir}")
audit_display.round(3)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


[Data] train: 17 classes, 3365 images
[Data] val: 17 classes, 903 images
[Data] test: 17 classes, 532 images
Audit device: cuda
Isolated output: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/Task-B-new/swin_val100_recheck_v1
Existing runs in this audit namespace resume/reuse; official checkpoints are never touched.
Reference metrics: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/phase_1/final_b_recomputed/split_metrics.csv

Fresh audit config: main_seed123
[recheck_main_seed123 | swin_tiny s123] train=3365, val=903, test=532, classes=17


wandb: Currently logged in as: ny0g507h3p (ny0g507h3p_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Epoch 1/50 | Train Loss: 2.8513, Acc: 7.95% | Val Loss: 2.8501, Acc: 6.98% | LR: 2.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed123_best.pth (Val Acc: 6.98%)


Epoch 2/50 | Train Loss: 1.6703, Acc: 64.76% | Val Loss: 0.8201, Acc: 93.02% | LR: 4.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed123_best.pth (Val Acc: 93.02%)


Epoch 3/50 | Train Loss: 0.7649, Acc: 96.28% | Val Loss: 0.6794, Acc: 96.79% | LR: 6.00e-05[Best][EarlyStop 0/10]
  ↳ New best! Saved to swin_tiny_seed123_best.pth (Val Acc: 96.79%)


... (56 outputs left out)

Epoch 20/50 | Train Loss: 1.0138, Acc: 99.79% | Val Loss: 1.0203, Acc: 99.22% | LR: 7.50e-05[EarlyStop 10/10]

[EarlyStopping] No improvement for 10 epochs. Stopping at epoch 20.


best_epoch,▁
best_val_acc,▁
epochs_trained,▁
lr,▁▃▅▆███████▇▇▇▇▇▇▆▆▆
test/accuracy,▁
test/f1/Araucaria araucana,▁
test/f1/Callitris columellaris,▁
test/f1/Calocedrus rupestris,▁
test/f1/Chamaecyparis obtusa,▁
test/f1/Cryptomeria japonica,▁
+19,...


[recheck_label_smoothing_0.2_seed42 | swin_tiny s42] val_acc=99.78% test_acc=98.87% F1=98.96%
Saved audit comparison: /home/sakana/Code/ManifoldsTeam/woodbench/data/experiments/Task-B-new/swin_val100_recheck_v1


,config,seed,reported_best_val_accuracy_pct,best_checkpoint_val_accuracy,best_checkpoint_test_accuracy,best_checkpoint_test_f1_macro,val_test_accuracy_gap_pp,test_accuracy_delta_vs_reference_pp
0,main_seed123,123,99.779,99.779,99.624,99.654,0.154,-0.188
1,rotation_90_seed42,42,99.779,99.779,99.812,99.827,-0.034,0.000
2,label_smoothing_0.2_seed42,42,99.779,99.779,99.248,99.310,0.530,-0.376
